# Lab Work - 4.5

# Q1. Bootstrap Sampling

In [ ]:
import numpy as np
import pandas as pd

data = pd.DataFrame({
    'Index':[1,2,3,4,5,6],
    'x':[1,2,3,4,5,6],
    'y':[10,20,25,28,40,45]
})

data

## Bootstrap Sample B1

In [ ]:
B1 = [2,4,2,6,1,5]
sample_B1 = data.iloc[np.array(B1)-1]
sample_B1

## Bootstrap Sample B2

In [ ]:
B2 = [3,3,5,1,4,6]
sample_B2 = data.iloc[np.array(B2)-1]
sample_B2

## Bootstrap Sample B3

In [ ]:
B3 = [6,2,4,4,1,3]
sample_B3 = data.iloc[np.array(B3)-1]
sample_B3

In [ ]:
original = set([1,2,3,4,5,6])

def get_oob(sample):
    return sorted(list(original - set(sample)))

oob_B1 = get_oob(B1)
oob_B2 = get_oob(B2)
oob_B3 = get_oob(B3)

print('OOB B1:', oob_B1)
print('OOB B2:', oob_B2)
print('OOB B3:', oob_B3)

In [ ]:
for name,oob in zip(['B1','B2','B3'],[oob_B1,oob_B2,oob_B3]):
    print(name,'OOB Count =',len(oob))

Theoretical OOB Probability:

$$(1 - 1/n)^n \approx e^{-1} \approx 0.368$$

Thus approximately 36.8% of samples are OOB and 64.2% appear in a bootstrap sample.

# Q2. Feature Randomness and Tree Construction

In [ ]:
from sklearn.tree import DecisionTreeRegressor

X = data[['x']]
y = data['y']

In [ ]:
tree1 = DecisionTreeRegressor(max_depth=2, random_state=1)
tree2 = DecisionTreeRegressor(max_depth=2, random_state=2)
tree3 = DecisionTreeRegressor(max_depth=2, random_state=3)

tree1.fit(sample_B1[['x']], sample_B1['y'])
tree2.fit(sample_B2[['x']], sample_B2['y'])
tree4.fit(sample_B3[['x']], sample_B3['y'])


In [ ]:
print('Tree 1 Depth:', tree1.get_depth())
print('Tree 2 Depth:', tree2.get_depth())
print('Tree 3 Depth:', tree4.get_depth())

Different bootstrap samples create different split locations and leaf means even when using the same original dataset.

# OOB MSE for Individual Trees

In [ ]:
def oob_mse(tree,oob_indices):

    if len(oob_indices)==0:
        return None

    subset = data[data['Index'].isin(oob_indices)]

    pred = tree.predict(subset[['x']])

    mse = np.mean((subset['y'] - pred)**2)

    return mse

print('Tree1 OOB MSE:', oob_mse(tree1,oob_B1))
print('Tree2 OOB MSE:', oob_mse(tree2,oob_B2))
print('Tree3 OOB MSE:', oob_mse(tree3,oob_B3))

# Q4. Aggregate Predictions

In [ ]:
ensemble_predictions = []

for value in X.values:

    p1 = tree1.predict([value])[0]
    p2 = tree2.predict([value])[0]
    p3 = tree4.predict([value])[0]

    avg = (p1+p2+p3)/3

    ensemble_predictions.append(avg)

ensemble_predictions

In [ ]:
ensemble_mse = np.mean((y - ensemble_predictions)**2)

print('Ensemble Training MSE =', ensemble_mse)

In [ ]:
result = pd.DataFrame({
    'x':X['x'],
    'Actual':y,
    'Ensemble Prediction':ensemble_predictions
})

result

# OOB Ensemble Prediction

In [ ]:
oob_predictions = {}

for idx in range(1,7):
    preds = []

    if idx in oob_B1:
        preds.append(tree1.predict([[idx]])[0])

    if idx in oob_B2:
        preds.append(tree2.predict([[idx]])[0])

    if idx in oob_B3:
        preds.append(tree4.predict([[idx]])[0])

    if len(preds)>0:
        oob_predictions[idx] = np.mean(preds)

oob_predictions

In [ ]:
errors=[]

for idx,pred in oob_predictions.items():

    actual = data.loc[data['Index']==idx,'y'].values[0]

    errors.append((actual-pred)**2)

oob_mse = np.mean(errors)

print('Overall OOB MSE =', oob_mse)

OOB MSE is often a better estimate of generalization performance because predictions are made using trees that never saw those observations during training.

# Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.scatter(X,y,label='Actual Data')
plt.plot(X,ensemble_predictions,label='Ensemble Prediction')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Random Forest Ensemble Prediction')
plt.legend()
plt.grid(True)
plt.show()

# Q4. Theory Questions

## Q4.1 Random Forest Prediction Rule

$$
\hat{y}_{RF}(x)=\frac{1}{B}\sum_{b=1}^{B}\hat{y}_b(x)
$$

The final prediction is the average prediction from all trees.

## Q4.2 Bias-Variance Reduction

Averaging multiple weakly correlated trees greatly reduces variance while maintaining similar bias.

Variance of an average decreases approximately by a factor of 1/B when trees are independent.

## Q4.3 Role of Feature Randomness

Instead of examining all p features, each split considers only m features.

This decorrelates trees, increases diversity, and improves ensemble performance.

## Q4.4 Out-of-Bag Error

OOB Error is calculated using observations not included in the bootstrap sample.

Since those observations were never used for training, OOB error behaves similarly to validation error and estimates generalization performance without requiring a separate validation set.

# Conclusion

- Bootstrap sampling creates diverse datasets.
- Different samples produce different trees.
- Averaging tree predictions reduces variance.
- OOB error provides a built-in estimate of test performance.
- Random Forest combines bootstrap sampling and feature randomness to improve prediction accuracy.